# Planner형 Agent: 기술 조사 보고서

PydanticAI를 활용하여 웹 검색과 섹션 작성 도구를 **반복 호출**하며 기술 조사 보고서를 자율적으로 작성하는 **Planner형 Agent**입니다.

**Planner형 Agent의 특징:**
- LLM이 스스로 **계획을 수립**하고 도구 호출 순서를 결정
- 한 번의 요청으로 **여러 도구를 반복 호출** (검색 → 작성 → 검색 → 작성 ...)
- 자동화형과 달리 **호출 횟수와 순서가 고정되지 않음** — LLM이 판단

## 환경 설정

`.env` 파일에서 API 키를 로드하고, PydanticAI Agent를 임포트합니다.

In [ ]:
# .env 파일에서 OPENAI_API_KEY 등 환경변수 로드
from dotenv import load_dotenv
load_dotenv()

from pydantic_ai import Agent  # LLM 기반 Agent 프레임워크

## Agent 정의

Planner형 Agent는 `output_type`을 지정하지 않고 **자유 텍스트**를 반환합니다.
instructions에 도구 사용 전략(검색 → 섹션 작성 → 반복)을 명시하여 LLM이 자율적으로 계획을 세우도록 유도합니다.

In [ ]:
# Planner형: output_type 없이 자유 텍스트 반환
# instructions로 도구 사용 전략을 안내 (LLM이 자율적으로 계획)
agent = Agent(
    "openai:gpt-5.4",
    instructions=(
        "기술 조사 보고서를 작성하는 리서치 Agent.\n"
        "1. search_web으로 필요한 정보를 조사 (2-3회)\n"
        "2. write_section으로 보고서 섹션 작성\n"
        "3. 최소 3개 섹션: 개요, 주요 비교, 결론/추천\n"
        "모든 내용은 한국어. 모든 섹션 작성 후 최종 요약을 반환."
    ),
)

## 보고서 저장소

Agent가 `write_section` Tool을 호출할 때마다 여기에 섹션이 추가됩니다.
Agent 실행 후 이 리스트를 순회하면 완성된 보고서를 확인할 수 있습니다.

In [ ]:
# Agent의 write_section Tool이 섹션을 누적 저장하는 리스트
report_sections: list[dict[str, str]] = []

## Tools (실제 API + Fallback)

DuckDuckGo 검색 API와 파일 저장을 사용하는 **실제 도구**를 정의합니다.
라이브러리가 없거나 검색에 실패하면 자동으로 **Fallback 데이터**를 반환하므로,
환경 설정 없이도 실습이 가능합니다.

- `search_web`: DuckDuckGo 검색 API로 실시간 웹 검색 (`duckduckgo-search` 패키지 필요)
- `write_section`: 보고서 섹션을 메모리와 `report_output.md` 파일에 동시 저장

`@agent.tool_plain`은 Agent 컨텍스트 없이 독립적으로 동작하는 도구를 정의하는 데코레이터입니다.

In [ ]:
# 검색 도구: DuckDuckGo API로 실시간 웹 검색 (실패 시 Fallback)
@agent.tool_plain
def search_web(query: str) -> str:
    """웹에서 정보를 검색한다. 기술 동향, 사례, 통계 조사용."""
    print(f"  [Tool] search_web('{query}')")
    try:
        from ddgs import DDGS
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=3))
        if results:
            lines = []
            for i, r in enumerate(results, 1):
                lines.append(f"{i}. {r['title']}: {r['body'][:150]}")
            print(f"  [Tool] → {len(results)}건 검색 완료 (DuckDuckGo)")
            return f"'{query}' 검색 결과:\n" + "\n".join(lines)
    except Exception as e:
        print(f"  [Tool] → Fallback 사용 ({e})")
    return (
        f"'{query}' 검색 결과:\n"
        "1. 해당 기술의 도입률이 전년 대비 45% 증가.\n"
        "2. Google, Meta, OpenAI가 관련 투자 확대 중.\n"
        "3. 오픈소스 생태계 성장, GitHub 스타 합계 10만 돌파."
    )


# 섹션 작성 도구: 메모리 저장 + 실제 마크다운 파일에도 기록
@agent.tool_plain
def write_section(title: str, content: str) -> str:
    """보고서의 한 섹션을 작성하여 저장한다."""
    from pathlib import Path
    report_sections.append({"title": title, "content": content})
    # 실제 마크다운 파일에도 기록
    report_file = Path("report_output.md")
    with open(report_file, "a", encoding="utf-8") as f:
        f.write(f"\n## {title}\n\n{content}\n")
    print(f"  [Tool] write_section('{title}') → 섹션 #{len(report_sections)} (report_output.md에 저장)")
    return f"섹션 '{title}' 저장 완료 (report_output.md)"

## 실행

Agent에게 주제를 전달하면, LLM이 자율적으로 검색과 섹션 작성을 반복합니다.
Tool 호출 로그를 통해 Agent가 어떤 순서로 계획을 실행하는지 관찰할 수 있습니다.

> **참고:** Jupyter 노트북은 이미 이벤트 루프가 실행 중이므로 `run_sync()` 대신 `await agent.run()`을 사용합니다.

In [ ]:
# 조사할 주제 설정
topic = "AI Agent 프레임워크 비교 (LangGraph vs CrewAI vs PydanticAI)"
print(f"주제: {topic}\n")

# Agent 실행 — LLM이 search_web/write_section을 자율적으로 반복 호출
# Jupyter에서는 await 사용 (run_sync는 이벤트 루프 충돌 발생)
result = await agent.run(f"다음 주제로 기술 조사 보고서를 작성해주세요: {topic}")

### 기술 조사 보고서 출력

Agent가 `write_section` Tool로 저장한 섹션들을 순서대로 출력합니다.

In [ ]:
# write_section Tool이 저장한 보고서 섹션을 순회하며 출력
for sec in report_sections:
    print(f"\n## {sec['title']}\n{sec['content']}")

### Agent 최종 응답

Agent가 모든 도구 호출을 마친 후 반환한 최종 요약 텍스트입니다.
`output_type`을 지정하지 않았으므로 자유 형식의 문자열이 반환됩니다.

In [ ]:
# Agent의 최종 응답 (자유 텍스트)
print(result.output)